In [2]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ishwa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ishwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
# Install required packages
!pip install tweepy vaderSentiment

import tweepy
import pandas as pd
import time
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# API credentials
consumer_key = "b9alPDmhxGER3q7zxLjHKrM2K"
consumer_secret = "LUmttAbRdfvMHFIzpGV2rqmJRMczTrd3itk9pMDpaaV1LNfBE3"
access_token = "1929867661061902339-TiLV0aUvJoKnINp9fVOn4fT5eETZDV"
access_token_secret = "5zXcIPLUM7ARYxfeOtAwqqYJNw2ycbi9Wt46KaDpIxsth"
bearer_token = "AAAAAAAAAAAAAAAAAAAAAAXb2AEAAAAArstZ2Oe4OEpXGCskUW29zMqs9vs%3DgiabRoegJrwlo9ktSUEznP6K0VEOHUyrRV1vG0a2qsq1fU2Wsm"  

client = tweepy.Client(
    bearer_token=bearer_token,
    consumer_key=consumer_key, 
    consumer_secret=consumer_secret,
    access_token=access_token, 
    access_token_secret=access_token_secret
)

query = "(Apple OR Microsoft) -is:retweet"
tweets = []
limit = 10  # Reduced from 100 to avoid rate limits

try:
    # Adding error handling and rate limit awareness
    response = client.search_recent_tweets(
        query=query,
        max_results=limit,
        tweet_fields=['created_at', 'text']
    )
    
    if response.data:
        for tweet in response.data:
            tweets.append([tweet.created_at, tweet.text])
    
    # Create DataFrame
    df = pd.DataFrame(tweets, columns=["date", "text"])
    
    # If we have tweets, perform sentiment analysis
    if not df.empty:
        # Initialize sentiment analyzer
        analyzer = SentimentIntensityAnalyzer()
        
        # Calculate sentiment scores
        df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
        
        # Classify sentiment
        df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
        
        # Display results
        print(df[['date', 'text', 'sentiment', 'label']].head())
    else:
        print("No tweets were retrieved. Creating a sample DataFrame for demonstration.")
        # Create a sample DataFrame if no tweets were retrieved
        sample_data = {
            'date': ['2023-01-01', '2023-01-02', '2023-01-03'],
            'text': [
                'I love the new Apple products!', 
                'Microsoft services are down again, very frustrating.', 
                'The new tech releases are okay I guess.'
            ]
        }
        df = pd.DataFrame(sample_data)
        
        # Initialize sentiment analyzer
        analyzer = SentimentIntensityAnalyzer()
        
        # Calculate sentiment scores
        df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
        
        # Classify sentiment
        df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
        
        # Display results
        print("Sample data sentiment analysis:")
        print(df[['date', 'text', 'sentiment', 'label']])
    
except tweepy.TooManyRequests:
    print("Rate limit exceeded. Using sample data instead.")
    # Create a sample DataFrame if rate limited
    sample_data = {
        'date': ['2023-01-01', '2023-01-02', '2023-01-03'],
        'text': [
            'I love the new Apple products!', 
            'Microsoft services are down again, very frustrating.', 
            'The new tech releases are okay I guess.'
        ]
    }
    df = pd.DataFrame(sample_data)
    
    # Initialize sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()
    
    # Calculate sentiment scores
    df['sentiment'] = df['text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
    
    # Classify sentiment
    df['label'] = df['sentiment'].apply(lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral'))
    
    # Display results
    print("Sample data sentiment analysis:")
    print(df[['date', 'text', 'sentiment', 'label']])
    
except Exception as e:
    print(f"An error occurred: {e}")

                       date  \
0 2025-06-03 12:39:24+00:00   
1 2025-06-03 12:39:22+00:00   
2 2025-06-03 12:39:20+00:00   
3 2025-06-03 12:39:17+00:00   
4 2025-06-03 12:39:16+00:00   

                                                text  sentiment     label  
0  Linee 4-50-77-83-86: Modifica temporanea perco...     0.3400  positive  
1  Dm me with your cashtag,PayPal,Venmo,Apple Pay...     0.0000   neutral  
2  生真面目刑事と秘密の取調べ中♡SynClub（シンクラブ）DLリンク：https://t.c...     0.0000   neutral  
3  @factualmama In baking, apple puree can work w...     0.2732  positive  
4  Not good for $GOOGL for sure\n\nApple $AAPL re...     0.4007  positive  


In [12]:
import re

# Add company label from text
def extract_company(text):
    text = text.lower()
    if 'apple' in text:
        return 'Apple'
    elif 'microsoft' in text:
        return 'Microsoft'
    else:
        return 'Unknown'

df['company'] = df['text'].apply(extract_company)

# Convert datetime to just date
df['date'] = pd.to_datetime(df['date']).dt.date

# Group by company and date, then take mean sentiment
daily_sentiment = df.groupby(['company', 'date'])['sentiment'].mean().reset_index()
daily_sentiment.rename(columns={'sentiment': 'avg_sentiment'}, inplace=True)

print(daily_sentiment.head())


     company        date  avg_sentiment
0      Apple  2025-06-03       0.168983
1  Microsoft  2025-06-03       0.000000
2    Unknown  2025-06-03       0.120400


In [3]:
import os
print(os.getcwd())


C:\Users\ishwa\anaconda_projects\db


In [8]:

# __file__ is not available in Jupyter notebooks because notebooks
# don't have a traditional file path like Python scripts
# Here's a workaround to get the notebook's path

import os
from IPython import get_ipython

# Method 1: Get the current working directory
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

# Method 2: If you need the notebook name (requires notebook to be saved)
try:
    notebook_path = get_ipython().kernel.session.config.get('notebook_path', '')
    if notebook_path:
        print(f"Notebook path: {notebook_path}")
    else:
        print("Notebook path not available. Make sure the notebook is saved.")
except:
    print("Could not determine notebook path")

Current working directory: C:\Users\ishwa\anaconda_projects\db
Notebook path not available. Make sure the notebook is saved.


'D:\\Anaconda\\envs\\sentiment-quant\\lib\\site-packages\\notebook\\__init__.py'

In [10]:
import os
os.getcwd()



'C:\\Users\\ishwa\\anaconda_projects\\db'